# <b>Training the MLP Model</b>

In this notebook, we're going to train the `MultiLayerPerceptron` that we've implemented with PyTorch and see how it performs before we do any tweaking on anything, neither the architecture, neural network layers, initialisation, the datasets or hyperparameters. Obviously we'll have a bad model as layers like CNNs perform better with images, but let's go through the process nonetheless. It's good for experience.

In [1]:
import os
print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: c:\Users\kmahl\Documents\ai-eng\personal-projects\learning-pytorch\notebooks


In [2]:
wsl_path = "/root/Documents/ai-eng/learning-pytorch"
windows_path = "c:/Users/kmahl/Documents/ai-eng/personal-projects/learning-pytorch"

os.chdir(windows_path)
print(os.getcwd())

c:\Users\kmahl\Documents\ai-eng\personal-projects\learning-pytorch


In [15]:
import torch
import torch.nn as nn
import torch.utils.data as data

from data.pytorchdata.catvnoncat import CatVsNonCatDataset

from models.MultiLayerPerceptron import MultiLayerPerceptron
from trainer.MLPTrainer import MLPTrainer
from models.model_visuals import plot_metric

import h5py
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## <b>Explore the dataset</b>

In [5]:
train_file_path = "data/rawdata/catvnoncat/train_catvnoncat.h5"
test_file_path = "data/rawdata/catvnoncat/test_catvnoncat.h5"

def explore_dataset(file_path, dataset_name, dataset_type):
  with h5py.File(file_path, "r") as file:
    print(f"Exploring {dataset_name} dataset:")
    
    print("Keys in the file:", list(file.keys()))
    
    images = file[f"{dataset_type}_set_x"][()]
    labels = file[f"{dataset_type}_set_y"][()]
    
    print(f"Number of images: {images.shape[0]}")
    print(f"Image shape: {images.shape[1:]}")
    print(f"Number of labels: {labels.shape[0]}")
    
    unique_labels, counts = np.unique(labels, return_counts=True)
    for label, count in zip(unique_labels, counts):
      print(f"Label {label}: {count} images")

In [6]:
explore_dataset(train_file_path, "Training", "train")

Exploring Training dataset:
Keys in the file: ['list_classes', 'train_set_x', 'train_set_y']
Number of images: 209
Image shape: (64, 64, 3)
Number of labels: 209
Label 0: 137 images
Label 1: 72 images


In [7]:
explore_dataset(test_file_path, "Test", "test")

Exploring Test dataset:
Keys in the file: ['list_classes', 'test_set_x', 'test_set_y']
Number of images: 50
Image shape: (64, 64, 3)
Number of labels: 50
Label 0: 17 images
Label 1: 33 images


In [ ]:
# with h5py.File(train_file_path, "r") as file:
#   images = file["train_set_x"][()]
#   labels = file["train_set_y"][()]

# cat_index = np.where(labels == 1)[0][0]
# non_cat_index = np.where(labels == 0)[0][0]

# fig = make_subplots(rows=1, cols=2, subplot_titles=("Cat", "Non-Cat"))

# fig.add_trace(go.Image(z=images[cat_index]), row=1, col=1)
# fig.add_trace(go.Image(z=images[non_cat_index]), row=1, col=2)

# fig.update_layout(title_text="Example of a Cat and a Non-Cat", title_x=0.5)
# fig.update_xaxes(showticklabels=False)
# fig.update_yaxes(showticklabels=False)

# fig.show()

## <b>Training the model</b>

In [8]:
train_dataset = CatVsNonCatDataset(train_file_path, train=True)
test_dataset = CatVsNonCatDataset(test_file_path, train=False)

In [19]:
input_size = 64 * 64 * 3  # 12288 pixels per image
output_size = 1  # Binary classification

# hyperparams
epochs = 40
batch_size = 16
learning_rate = 0.001
total_steps = 500

model = MultiLayerPerceptron(
  layer_dims=[input_size, 128, 64, output_size], 
  activations=[torch.relu, torch.relu]
)

In [20]:
train_loader = data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [21]:
trainer = MLPTrainer(model, train_loader, val_loader, loss_fn, optimizer, epochs=epochs)
trainer.train_steps(total_steps=total_steps)

In [22]:
plot_metric(trainer.train_losses, trainer.val_losses, metric_name="Loss")

In [23]:
plot_metric(trainer.train_errors, trainer.val_errors, metric_name="Error Rate")